In [3]:
import sys
from pathlib import Path

import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio

if 'vscode' in pio.renderers:
    pio.renderers.default = 'vscode'
else:
    pio.renderers.default = 'notebook'

cwd = Path.cwd().resolve()
repo_candidates = [cwd, *cwd.parents]
repo_root = next((p for p in repo_candidates if (p / 'src').exists() and (p / 'data').exists()), cwd)
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

clean_path = repo_root / 'data' / 'processed' / 'ibrd_clean.csv'
if not clean_path.exists():
    from src.data_pipeline.silver_layer import clean_ibrd_data
    clean_ibrd_data(
        raw_path=repo_root / 'data' / 'raw' / 'ibrd_synthetic.csv',
        silver_path=clean_path,
    )

df = pd.read_csv(clean_path)
if 'Board Approval Date' in df.columns:
    df['Board Approval Date'] = pd.to_datetime(df['Board Approval Date'], errors='coerce')
if 'loan_age_years' not in df.columns:
    approval_dates = pd.to_datetime(df['Board Approval Date'], errors='coerce')
    df['loan_age_days'] = (pd.Timestamp.now() - approval_dates).dt.days
    df['loan_age_years'] = df['loan_age_days'] / 365.25

output_dir = repo_root / 'notebooks' / 'exports'
output_dir.mkdir(parents=True, exist_ok=True)


def show_and_save(fig, filename):
    fig.show(renderer='vscode')
    try:
        export_path = output_dir / f'{filename}.png'
        fig.write_image(export_path, width=1400, height=900, scale=2)
        print(f'Saved chart: {export_path}')
    except Exception as exc:
        print(f'PNG export skipped: {exc}')

# Annual lending activity
if 'Board Approval Date' in df.columns:
    df['approval_year'] = df['Board Approval Date'].dt.year
    approval_by_year = df.groupby('approval_year').agg({
        'Original Principal Amount (US$)': 'sum',
        'Loan Number': 'count'
    }).reset_index()
    approval_by_year.columns = ['Year', 'Total Commitments', 'Number of Loans']

    fig = make_subplots(specs=[[{'secondary_y': True}]])
    fig.add_trace(go.Bar(x=approval_by_year['Year'], y=approval_by_year['Number of Loans'], name='Number of Loans', marker_color='lightblue'), secondary_y=False)
    fig.add_trace(go.Scatter(x=approval_by_year['Year'], y=approval_by_year['Total Commitments'] / 1e9, name='Commitments (Billion $)', marker_color='darkblue'), secondary_y=True)
    fig.update_layout(title='Annual Lending Activity', xaxis_title='Year')
    show_and_save(fig, 'annual_lending_activity')

# Portfolio aging
if 'loan_age_years' in df.columns:
    df['age_category'] = pd.cut(
        df['loan_age_years'],
        bins=[0, 5, 10, 20, 30, 50, 100],
        labels=['0-5', '5-10', '10-20', '20-30', '30-50', '50+'],
    )
    age_summary = df.groupby('age_category').agg({
        'Original Principal Amount (US$)': 'sum',
        'Due to IBRD (US$)': 'sum',
        'Loan Number': 'count'
    }).reset_index()

    fig = px.bar(
        age_summary,
        x='age_category',
        y='Due to IBRD (US$)',
        title='Outstanding Amount by Loan Age',
        labels={'age_category': 'Loan Age (Years)', 'Due to IBRD (US$)': 'Outstanding Amount'},
    )
    show_and_save(fig, 'outstanding_amount_by_loan_age')

Wait expired, Browser is being closed by watchdog.


Saved chart: /home/rigii/ATA/notebooks/exports/annual_lending_activity.png


Saved chart: /home/rigii/ATA/notebooks/exports/outstanding_amount_by_loan_age.png
